In [1]:
# Import necessary packages (might need to download some in the terminal)
import os
import re
import pandas as pd
import nltk
import pdfplumber

# Define input directory containing CAAR market report files
DATA_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 2/caar_market_reports"
# Define output path for the structured dashboard metrics CSV
OUTPUT_CSV = "/Users/emilymoore/Downloads/DS 4002 Project 2/caar_market_reports/caar_data.csv"

# Define regex patterns to extract structured dashboard metrics
# Each key represents a variable name in the dataset
# Each value is a regular expression pattern that captures the numeric value
patterns = {
    "sales_volume": r"([\d,]+)\s+Sales",
    "median_price": r"\$([\d,]+)\s+Median Sales Price"
}

rows = [] # Initialize an empty list to store extracted rows

# Loop through every file in the CAAR reports directory
for file in os.listdir(DATA_DIR):
    file_path = os.path.join(DATA_DIR, file)
    full_text = ""
    
    # If file is a PDF, extract text using pdfplumber
    if file.lower().endswith(".pdf"):
        with pdfplumber.open(file_path) as pdf:
            full_text = " ".join(page.extract_text() or "" for page in pdf.pages)
    # If file is a TXT file, read directly 
    elif file.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            full_text = f.read()
    else:
        continue # Skip files that are neither PDF nor TXT

    # Create a dictionary for this file’s extracted metrics
    row = {"file_name": file}

    # Search for each metric using its regex pattern
    for field, pattern in patterns.items():
        match = re.search(pattern, full_text, re.IGNORECASE)
        if match:
            value = match.group(1).replace(",", "") # Remove commas from numeric values
            row[field] = value
        else:
            row[field] = None # If metric not found in file, store as None

    rows.append(row) # Append the extracted row to the list

df_caar = pd.DataFrame(rows) # Convert list of dictionaries into a structured DataFrame
df_caar.to_csv(OUTPUT_CSV, index=False) # Save the extracted market metrics to CSV for analysis

print(f"Saved the reports to {OUTPUT_CSV}")

Saved the reports to /Users/emilymoore/Downloads/DS 4002 Project 2/caar_market_reports/caar_data.csv
